# **Filtrado de Investigations por Exclusión de Marcas C/E**

Este notebook filtra los datos de Investigations excluyendo las marcas que son exclusivas de Child Seats (C) y Equipment (E).

**Objetivos:**
1. Cargar lista de marcas exclusivas (727 marcas identificadas en Chunk 1)
2. Filtrar Investigations excluyendo esas marcas
3. Guardar archivo filtrado

In [1]:
import pandas as pd
from pathlib import Path
import json

# Configuración de rutas
BASE_DIR = Path(".." if Path.cwd().name == "notebooks" else ".")
IN_DIR = BASE_DIR / "data/processed"
OUT_DIR = BASE_DIR / "data/processed"

print(f"Working directory: {Path.cwd()}")
print(f"Base directory: {BASE_DIR}")

Working directory: c:\Users\moral\tec_final\notebooks
Base directory: ..


In [2]:
# PASO 1: Cargar marcas excluibles
print("="*70)
print("PASO 1: CARGA DE MARCAS EXCLUIBLES")
print("="*70)

with open(OUT_DIR / "excluded_makes_from_c_e.json", "r") as f:
    excluded_makes = set(json.load(f))

print(f"[OK] Marcas excluibles: {len(excluded_makes)}")
print(f"\nPrimeras 20 marcas excluibles:")
print(sorted(list(excluded_makes))[:20])

PASO 1: CARGA DE MARCAS EXCLUIBLES
[OK] Marcas excluibles: 727

Primeras 20 marcas excluibles:
['20TH CENTURY', '3M', '4WHEELS PARTS', 'A-1 ALTERNATIVE FUEL SYST', 'A.L. SOLUTIONS', 'A2ZEV', 'AAC', 'AAI MOTORSPORTS', 'ABADDON PRODUCTS', 'ABS-ACT-35 3-PT LWM 12', 'AC DELCO', 'ACC', 'ACCESSORY', 'ACCESSORY DISTRIBUTORS', 'ACCU-FAB', 'ACCU-FLOW', 'ACCURIDE', 'ACD TRIDON', 'ACE', 'ACE ELECTRIC']


In [4]:
# PASO 2: Cargar y filtrar INVESTIGATIONS
print("\n" + "="*70)
print("PASO 2: FILTRADO DE INVESTIGATIONS")
print("="*70)

df_investigations = pd.read_parquet(IN_DIR / "investigations.parquet")
print(f"Investigations originales: {len(df_investigations):,}")
print(f"Columnas: {df_investigations.columns.tolist()}")

# Normalizar marca
df_investigations['MAKE_NORMALIZED'] = df_investigations['MAKE'].astype(str).str.strip().str.upper()

# Verificación: cuántas marcas C/E hay originalmente
makes_in_data = set(df_investigations['MAKE_NORMALIZED'].unique())
matches = makes_in_data & excluded_makes
print(f"\nMarcas C/E en datos originales: {len(matches)}")
print(f"Marcas C/E encontradas: {sorted(list(matches))[:20] if matches else 'Ninguna'}")

# Filtrar
df_investigations_filtered = df_investigations[~df_investigations['MAKE_NORMALIZED'].isin(excluded_makes)].copy()
df_investigations_filtered = df_investigations_filtered.drop(columns=['MAKE_NORMALIZED'])

print(f"\nInvestigations filtrados: {len(df_investigations_filtered):,}")
print(f"Investigations excluidos: {len(df_investigations) - len(df_investigations_filtered):,}")
print(f"Reducción: {100 * (len(df_investigations) - len(df_investigations_filtered)) / len(df_investigations):.1f}%")

# Verificación post-filtro
makes_filtered = set(df_investigations_filtered['MAKE'].astype(str).str.strip().str.upper().unique())
matches_filtered = makes_filtered & excluded_makes
print(f"\n[VERIFICACION] Marcas C/E en datos filtrados: {len(matches_filtered)}")
print(f"[VERIFICACION] {'EXITO: No quedan marcas C/E' if len(matches_filtered) == 0 else 'ERROR: Aun hay marcas C/E'}")


PASO 2: FILTRADO DE INVESTIGATIONS
Investigations originales: 146,595
Columnas: ['NHTSA ACTION NUMBER', 'MAKE', 'MODEL', 'YEAR', 'COMPNAME', 'MFR_NAME', 'ODATE', 'CDATE', 'CAMPNO', 'SUBJECT', 'SUMMARY', '__SOURCE_FILE__', 'ACTIONNUMBER', 'SUBJECT_LEN', 'SUMMARY_LEN', 'COMPNAME_LEN', 'TEXT_TOTAL_LEN']

Marcas C/E en datos originales: 43
Marcas C/E encontradas: ['ACCURIDE', 'AMERICAN WIRE WHEEL', 'ARA', 'ARROWCRAFT', 'BEAM', 'BIC', 'BRAKE PARTS', 'CATERPILLAR', 'CENTURY', 'COSCO', 'CUMMINS', 'CUSTOM AND COMMERCIAL', 'DANA', 'DJG', 'DOREL', 'DRIVE-MASTER', 'EATON', 'EDDIE BAUER', 'EIS', 'ENGINEERING COOLING']

Investigations filtrados: 146,068
Investigations excluidos: 527
Reducción: 0.4%

[VERIFICACION] Marcas C/E en datos filtrados: 0
[VERIFICACION] EXITO: No quedan marcas C/E


In [5]:
# PASO 3: Guardar
df_investigations_filtered.to_parquet(OUT_DIR / "investigations_filtered.parquet", index=False)

print("\n" + "="*70)
print("ARCHIVO GUARDADO")
print("="*70)
print(f"Investigations filtrados: {len(df_investigations_filtered):,} registros")
print(f"Archivo: {OUT_DIR / 'investigations_filtered.parquet'}")
print("\n" + "="*70)
print("FILTRADO DE INVESTIGATIONS COMPLETADO")
print("="*70)


ARCHIVO GUARDADO
Investigations filtrados: 146,068 registros
Archivo: ..\data\processed\investigations_filtered.parquet

FILTRADO DE INVESTIGATIONS COMPLETADO
